# Multimodal Fusion Training — Multi-Modal Disease Detection

**Self-contained Kaggle notebook** that trains the **cross-attention fusion** model
by combining a pre-trained image encoder and text encoder.

**Prerequisites (upload to Kaggle as datasets):**
- `image_encoder.pth`, `image_head.pth` from image_training notebook
- `text_encoder.pth`, `text_head.pth`, `tokenizer/` from text_training notebook

**Outputs:**
- `fusion_model.pth` (complete fused model state)
- `multimodal_config.json`
- Comparison table: image-only vs text-only vs multimodal

In [ ]:
!pip install -q transformers

In [ ]:
import os
import json
import random
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score
from tqdm.auto import tqdm
from PIL import Image
import torchvision
import torchvision.transforms as T

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Configuration

In [ ]:
CONFIG = {
    # Pre-trained encoder weights (separate Kaggle datasets)
    'image_pretrained_dir': '/kaggle/input/image-trained-model',
    'text_pretrained_dir': '/kaggle/input/text-trained-model',
    'output_dir': '/kaggle/working',

    # Dataset paths (Kaggle dataset mounts)
    'chest_data_dir': '/kaggle/input/data',
    'chest_csv': '/kaggle/input/data/Data_Entry_2017_v2020.csv',
    'skin_data_dir': '/kaggle/input/skin-cancer-mnist-ham10000',
    'skin_csv': '/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv',
    'brain_data_dir': '/kaggle/input/brain-tumor-classification-mri',

    # Architecture
    'image_backbone': 'efficientnet_b4',
    'image_feature_dim': 1792,
    'image_size': 380,
    'text_model_name': 'dmis-lab/biobert-v1.1',
    'text_hidden_dim': 768,
    'projection_dim': 512,
    'dropout': 0.3,
    'num_classes': 25,

    # Fusion
    'fusion_type': 'cross_attention',
    'fusion_num_heads': 8,
    'fusion_num_cross_layers': 2,

    # Training
    'batch_size': 16,
    'epochs': 15,
    'lr_fusion': 1e-4,
    'lr_encoder': 1e-5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'focal_alpha': 0.75,
    'focal_gamma': 2.0,
    'max_text_length': 128,
    'chest_sample_frac': 0.1,
    'val_split': 0.2,
    'seed': SEED,

    # Labels
    'all_labels': [
        'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration',
        'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax',
        'Consolidation', 'Edema', 'Emphysema', 'Fibrosis',
        'Pleural_Thickening', 'Hernia',
        'Melanoma', 'Melanocytic_Nevi', 'Basal_Cell_Carcinoma',
        'Actinic_Keratosis', 'Benign_Keratosis', 'Dermatofibroma',
        'Vascular_Lesion',
        'Glioma', 'Meningioma', 'Pituitary_Tumor', 'No_Tumor',
    ],
    'modality_index_ranges': {
        'chest': [0, 14],
        'skin': [14, 21],
        'brain': [21, 25],
    },
}

label_to_idx = {label: i for i, label in enumerate(CONFIG['all_labels'])}
print(f'Classes: {CONFIG["num_classes"]}')

## 2. Model Definitions (inline)

In [ ]:
# ── Image Encoder ────────────────────────────────────────────────────────

class ImageEncoder(nn.Module):
    def __init__(self, backbone='efficientnet_b4', feature_dim=1792,
                 projection_dim=512, dropout=0.3, pretrained=True):
        super().__init__()
        weights = 'IMAGENET1K_V1' if pretrained else None
        base = getattr(torchvision.models, backbone)(weights=weights)
        self.features = nn.Sequential(*list(base.children())[:-2])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.projection = nn.Sequential(
            nn.Linear(feature_dim, projection_dim),
            nn.LayerNorm(projection_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.spatial_projection = nn.Sequential(
            nn.Linear(feature_dim, projection_dim),
            nn.LayerNorm(projection_dim),
        )

    def forward(self, x):
        feat = self.features(x)
        pooled = self.pool(feat).flatten(1)
        return self.projection(pooled)

    def get_spatial_features(self, x):
        feat = self.features(x)  # (B, C, H, W)
        b, c, h, w = feat.shape
        spatial = feat.permute(0, 2, 3, 1).reshape(b, h * w, c)  # (B, HW, C)
        return self.spatial_projection(spatial)

    def get_feature_map(self, x):
        return self.features(x)


# ── Text Encoder ─────────────────────────────────────────────────────────

class TextEncoder(nn.Module):
    def __init__(self, model_name='dmis-lab/biobert-v1.1', hidden_dim=768,
                 projection_dim=512, dropout=0.3, max_length=128):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.projection_dim = projection_dim
        self.max_length = max_length
        self.bert = AutoModel.from_pretrained(model_name, output_attentions=True)
        self.projection = nn.Sequential(
            nn.Linear(hidden_dim, projection_dim),
            nn.LayerNorm(projection_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.last_attention_weights = None

    def _encode(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, output_attentions=True)
        self.last_attention_weights = outputs.attentions[-1].mean(dim=1).detach()
        return outputs.last_hidden_state

    def forward(self, input_ids, attention_mask):
        hidden = self._encode(input_ids, attention_mask)
        return self.projection(hidden[:, 0, :])

    def get_token_features(self, input_ids, attention_mask):
        hidden = self._encode(input_ids, attention_mask)
        return self.projection(hidden)


print('Encoder definitions done.')

In [ ]:
# ── Fusion Modules ────────────────────────────────────────────────────────

class CrossAttentionBlock(nn.Module):
    def __init__(self, dim=512, num_heads=8, dropout=0.1):
        super().__init__()
        self.cross_attn_img2txt = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.cross_attn_txt2img = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm1_img = nn.LayerNorm(dim)
        self.norm1_txt = nn.LayerNorm(dim)
        self.norm2_img = nn.LayerNorm(dim)
        self.norm2_txt = nn.LayerNorm(dim)
        self.ffn_img = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim * 4, dim), nn.Dropout(dropout))
        self.ffn_txt = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim * 4, dim), nn.Dropout(dropout))

    def forward(self, img_feats, txt_feats, txt_mask=None):
        key_padding_mask = ~txt_mask.bool() if txt_mask is not None else None
        h_img, _ = self.cross_attn_img2txt(img_feats, txt_feats, txt_feats, key_padding_mask=key_padding_mask)
        img_feats = self.norm1_img(img_feats + h_img)
        h_txt, _ = self.cross_attn_txt2img(txt_feats, img_feats, img_feats)
        txt_feats = self.norm1_txt(txt_feats + h_txt)
        img_feats = self.norm2_img(img_feats + self.ffn_img(img_feats))
        txt_feats = self.norm2_txt(txt_feats + self.ffn_txt(txt_feats))
        return img_feats, txt_feats


class CrossAttentionFusion(nn.Module):
    def __init__(self, dim=512, num_heads=8, num_layers=2, num_classes=25, dropout=0.3):
        super().__init__()
        self.layers = nn.ModuleList([CrossAttentionBlock(dim, num_heads, dropout=0.1) for _ in range(num_layers)])
        self.classifier = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, num_classes),
        )

    def forward(self, img_spatial, txt_tokens, txt_mask=None):
        for layer in self.layers:
            img_spatial, txt_tokens = layer(img_spatial, txt_tokens, txt_mask)
        img_pool = img_spatial.mean(dim=1)
        if txt_mask is not None:
            mask_f = txt_mask.unsqueeze(-1).float()
            txt_pool = (txt_tokens * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1)
        else:
            txt_pool = txt_tokens.mean(dim=1)
        fused = torch.cat([img_pool, txt_pool], dim=-1)
        return self.classifier(fused)


class ConcatFusion(nn.Module):
    def __init__(self, dim=512, num_classes=25, dropout=0.3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim // 2, num_classes),
        )

    def forward(self, img_emb, txt_emb):
        return self.classifier(torch.cat([img_emb, txt_emb], dim=-1))


# ── Full Multimodal Model ─────────────────────────────────────────────────

class MultiModalDetector(nn.Module):
    def __init__(self, image_encoder, text_encoder, fusion, num_classes=25,
                 projection_dim=512, dropout=0.3):
        super().__init__()
        self.image_encoder = image_encoder
        self.text_encoder = text_encoder
        self.fusion = fusion
        self.image_head = nn.Sequential(
            nn.Linear(projection_dim, projection_dim // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(projection_dim // 2, num_classes),
        )
        self.text_head = nn.Sequential(
            nn.Linear(projection_dim, projection_dim // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(projection_dim // 2, num_classes),
        )

    def forward(self, image=None, input_ids=None, attention_mask=None, mode='multimodal'):
        if mode == 'multimodal' and image is not None and input_ids is not None:
            img_spatial = self.image_encoder.get_spatial_features(image)
            txt_tokens = self.text_encoder.get_token_features(input_ids, attention_mask)
            return self.fusion(img_spatial, txt_tokens, attention_mask)
        elif mode == 'image_only' and image is not None:
            img_emb = self.image_encoder(image)
            return self.image_head(img_emb)
        elif mode == 'text_only' and input_ids is not None:
            txt_emb = self.text_encoder(input_ids, attention_mask)
            return self.text_head(txt_emb)
        else:
            raise ValueError(f'Invalid mode={mode} or missing inputs')


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        p = torch.sigmoid(inputs)
        ce = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        p_t = p * targets + (1 - p) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = alpha_t * (1 - p_t) ** self.gamma * ce
        return loss.mean()


print('All model definitions done.')

## 3. Symptom Generator (for pairing with images)

In [ ]:
# ── Inline symptom generator (same as text_training notebook) ──────────

DISEASE_SYMPTOM_MAP = {
    'Atelectasis': {'symptoms': ['reduced breath sounds on one side', 'mild dyspnea', 'low-grade fever', 'cough', 'tachypnea'], 'contexts': ['post-operative patient', 'prolonged bed rest'], 'severity_range': (2, 6)},
    'Cardiomegaly': {'symptoms': ['progressive dyspnea on exertion', 'orthopnea', 'bilateral lower extremity edema', 'fatigue', 'palpitations'], 'contexts': ['known history of hypertension', 'prior myocardial infarction'], 'severity_range': (4, 9)},
    'Effusion': {'symptoms': ['dyspnea that worsens when lying flat', 'pleuritic chest pain', 'reduced breath sounds at the base', 'dullness to percussion'], 'contexts': ['known malignancy', 'heart failure history'], 'severity_range': (3, 7)},
    'Infiltration': {'symptoms': ['productive cough', 'mild fever', 'fatigue', 'decreased appetite'], 'contexts': ['community-acquired infection', 'immunocompromised state'], 'severity_range': (2, 5)},
    'Mass': {'symptoms': ['persistent cough lasting more than 3 weeks', 'unintentional weight loss', 'hemoptysis', 'chest pain'], 'contexts': ['long history of smoking', 'occupational asbestos exposure'], 'severity_range': (4, 8)},
    'Nodule': {'symptoms': ['incidental finding on chest X-ray', 'no specific symptoms', 'mild persistent cough'], 'contexts': ['routine health screening', 'smoker over 50 years old'], 'severity_range': (1, 3)},
    'Pneumonia': {'symptoms': ['high fever with chills', 'productive cough with purulent sputum', 'severe dyspnea', 'pleuritic chest pain'], 'contexts': ['recent upper respiratory infection', 'elderly patient'], 'severity_range': (5, 9)},
    'Pneumothorax': {'symptoms': ['sudden onset sharp chest pain', 'acute dyspnea', 'reduced breath sounds on affected side', 'hypoxia'], 'contexts': ['tall thin young male', 'recent thoracic procedure'], 'severity_range': (6, 10)},
    'Consolidation': {'symptoms': ['productive cough', 'fever', 'bronchial breath sounds', 'increased tactile fremitus'], 'contexts': ['community-acquired pneumonia', 'aspiration event'], 'severity_range': (4, 7)},
    'Edema': {'symptoms': ['severe dyspnea at rest', 'orthopnea', 'pink frothy sputum', 'bilateral crackles'], 'contexts': ['acute heart failure exacerbation', 'fluid overload'], 'severity_range': (7, 10)},
    'Emphysema': {'symptoms': ['chronic progressive dyspnea', 'pursed lip breathing', 'barrel chest', 'prolonged expiratory phase'], 'contexts': ['long-term smoker', 'alpha-1 antitrypsin deficiency'], 'severity_range': (4, 8)},
    'Fibrosis': {'symptoms': ['progressive exertional dyspnea', 'dry non-productive cough', 'fine bibasilar crackles', 'digital clubbing'], 'contexts': ['idiopathic pulmonary fibrosis', 'occupational dust exposure'], 'severity_range': (4, 8)},
    'Pleural_Thickening': {'symptoms': ['mild chronic chest pain', 'decreased breath sounds at the base', 'mild dyspnea on exertion'], 'contexts': ['asbestos exposure history', 'previous pleural infection'], 'severity_range': (2, 5)},
    'Hernia': {'symptoms': ['epigastric fullness after meals', 'shortness of breath when supine', 'bowel sounds in the chest'], 'contexts': ['previous abdominal surgery', 'congenital diaphragmatic defect'], 'severity_range': (3, 7)},
    'Melanoma': {'symptoms': ['asymmetric mole with irregular borders', 'mole with multiple colors', 'recent change in size or color', 'itching or bleeding from a mole'], 'contexts': ['fair-skinned individual', 'family history of melanoma'], 'severity_range': (6, 10)},
    'Melanocytic_Nevi': {'symptoms': ['small round symmetric mole', 'uniform brown color', 'well-defined borders'], 'contexts': ['routine skin examination', 'cosmetic concern'], 'severity_range': (1, 2)},
    'Basal_Cell_Carcinoma': {'symptoms': ['pearly or waxy bump on the skin', 'bleeding or scabbing sore that keeps recurring', 'visible telangiectatic vessels'], 'contexts': ['sun-exposed area face neck', 'chronic sun exposure history'], 'severity_range': (3, 6)},
    'Actinic_Keratosis': {'symptoms': ['rough scaly patch on the skin', 'pink or red raised area', 'tenderness when touched'], 'contexts': ['chronic sun exposure', 'age over 50'], 'severity_range': (2, 5)},
    'Benign_Keratosis': {'symptoms': ['waxy stuck-on looking growth', 'well-circumscribed borders', 'brown or tan colored'], 'contexts': ['age-related skin change', 'trunk or back location'], 'severity_range': (1, 3)},
    'Dermatofibroma': {'symptoms': ['firm hard nodule under the skin', 'dimple sign when pinched', 'brownish or reddish color'], 'contexts': ['previous insect bite at the site', 'lower extremity location'], 'severity_range': (1, 3)},
    'Vascular_Lesion': {'symptoms': ['red or purple patch on the skin', 'lesion blanches with pressure', 'visible blood vessels'], 'contexts': ['present since birth or early childhood', 'cosmetic concern'], 'severity_range': (1, 3)},
    'Glioma': {'symptoms': ['progressive headaches worse in the morning', 'new onset seizures', 'focal neurological deficits', 'personality changes'], 'contexts': ['previously healthy adult', 'no prior history of seizures'], 'severity_range': (6, 10)},
    'Meningioma': {'symptoms': ['gradually worsening headaches', 'seizures', 'visual changes', 'weakness in extremities'], 'contexts': ['middle-aged female', 'history of cranial radiation'], 'severity_range': (3, 7)},
    'Pituitary_Tumor': {'symptoms': ['bitemporal hemianopsia', 'headaches', 'amenorrhea or irregular periods', 'galactorrhea'], 'contexts': ['young adult female', 'elevated prolactin levels'], 'severity_range': (3, 7)},
    'No_Tumor': {'symptoms': ['tension-type headache', 'migraine with aura', 'non-specific dizziness', 'no focal neurological deficits'], 'contexts': ['MRI ordered to rule out structural pathology', 'normal neurological exam'], 'severity_range': (1, 3)},
}

TEMPLATES = [
    'A {age}-year-old {gender} presents with {symptoms}. {context}. Symptoms present {duration}. Severity: {severity}.',
    'Patient is a {age}-year-old {gender} complaining of {symptoms} {duration}. Context: {context}. Severity: {severity}.',
    '{age} y/o {gender}. Chief complaint: {symptoms}. Duration: {duration}. Background: {context}. Severity: {severity}/10.',
]

DURATIONS = ['for the past few days', 'for about a week', 'for two weeks', 'for the past month', 'over the last several months']


def generate_symptom_text(disease, rng):
    info = DISEASE_SYMPTOM_MAP[disease]
    age = rng.randint(18, 85)
    gender = rng.choice(['male', 'female'])
    n = rng.randint(2, min(4, len(info['symptoms'])))
    syms = rng.sample(info['symptoms'], n)
    symptom_str = ', '.join(syms[:-1]) + f', and {syms[-1]}'
    sev = rng.randint(*info['severity_range'])
    return rng.choice(TEMPLATES).format(
        age=age, gender=gender, symptoms=symptom_str,
        context=rng.choice(info['contexts']),
        duration=rng.choice(DURATIONS), severity=sev
    )


print('Symptom generator ready.')

## 4. Data Loading (image + paired synthetic text)

In [ ]:
import pandas as pd
import glob


def load_chest_data(cfg, sample_frac=0.1):
    """Load NIH ChestX-ray14 subset."""
    df = pd.read_csv(cfg['chest_csv'])
    df = df.sample(frac=sample_frac, random_state=SEED)
    chest_labels = cfg['all_labels'][:14]
    records = []
    image_dirs = sorted(glob.glob(os.path.join(cfg['chest_data_dir'], 'images_*/images')))
    if not image_dirs:
        image_dirs = [os.path.join(cfg['chest_data_dir'], 'images')]
    for _, row in df.iterrows():
        fname = row['Image Index']
        path = None
        for d in image_dirs:
            candidate = os.path.join(d, fname)
            if os.path.exists(candidate):
                path = candidate
                break
        if path is None:
            continue
        findings = str(row['Finding Labels']).split('|')
        label_vec = np.zeros(cfg['num_classes'], dtype=np.float32)
        diseases_found = []
        for finding in findings:
            finding = finding.strip()
            if finding in label_to_idx and finding in chest_labels:
                label_vec[label_to_idx[finding]] = 1.0
                diseases_found.append(finding)
        if not diseases_found:
            continue
        records.append({
            'image_path': path, 'labels': label_vec,
            'diseases': diseases_found, 'modality': 'chest',
        })
    return records


def load_skin_data(cfg):
    """Load HAM10000."""
    skin_labels = cfg['all_labels'][14:21]
    dx_map = {
        'mel': 'Melanoma', 'nv': 'Melanocytic_Nevi', 'bcc': 'Basal_Cell_Carcinoma',
        'akiec': 'Actinic_Keratosis', 'bkl': 'Benign_Keratosis',
        'df': 'Dermatofibroma', 'vasc': 'Vascular_Lesion',
    }
    df = pd.read_csv(cfg['skin_csv'])
    img_dirs = [
        os.path.join(cfg['skin_data_dir'], 'HAM10000_images_part_1'),
        os.path.join(cfg['skin_data_dir'], 'HAM10000_images_part_2'),
        cfg['skin_data_dir'],
    ]
    records = []
    for _, row in df.iterrows():
        image_id = row['image_id']
        dx = row['dx']
        disease = dx_map.get(dx)
        if disease is None:
            continue
        path = None
        for d in img_dirs:
            candidate = os.path.join(d, f'{image_id}.jpg')
            if os.path.exists(candidate):
                path = candidate
                break
        if path is None:
            continue
        label_vec = np.zeros(cfg['num_classes'], dtype=np.float32)
        label_vec[label_to_idx[disease]] = 1.0
        records.append({
            'image_path': path, 'labels': label_vec,
            'diseases': [disease], 'modality': 'skin',
        })
    return records


def load_brain_data(cfg):
    """Load Brain Tumor MRI."""
    class_map = {
        'glioma_tumor': 'Glioma', 'meningioma_tumor': 'Meningioma',
        'pituitary_tumor': 'Pituitary_Tumor', 'no_tumor': 'No_Tumor',
    }
    records = []
    for split in ['Training', 'Testing']:
        split_dir = os.path.join(cfg['brain_data_dir'], split)
        if not os.path.isdir(split_dir):
            continue
        for cls_folder in os.listdir(split_dir):
            disease = class_map.get(cls_folder.lower())
            if disease is None:
                continue
            class_dir = os.path.join(split_dir, cls_folder)
            for fname in os.listdir(class_dir):
                if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue
                path = os.path.join(class_dir, fname)
                label_vec = np.zeros(cfg['num_classes'], dtype=np.float32)
                label_vec[label_to_idx[disease]] = 1.0
                records.append({
                    'image_path': path, 'labels': label_vec,
                    'diseases': [disease], 'modality': 'brain',
                })
    return records


# ── Load all ──────────────────────────────────────────────────────────────
print('Loading datasets...')
try:
    chest_records = load_chest_data(CONFIG, CONFIG['chest_sample_frac'])
except Exception as e:
    print(f'Chest data not available: {e}')
    chest_records = []

try:
    skin_records = load_skin_data(CONFIG)
except Exception as e:
    print(f'Skin data not available: {e}')
    skin_records = []

try:
    brain_records = load_brain_data(CONFIG)
except Exception as e:
    print(f'Brain data not available: {e}')
    brain_records = []

all_records = chest_records + skin_records + brain_records
print(f'\nTotal records: {len(all_records)}')
print(f'  Chest: {len(chest_records)}, Skin: {len(skin_records)}, Brain: {len(brain_records)}')

# Train/val split
modalities = [r['modality'] for r in all_records]
train_records, val_records = train_test_split(
    all_records, test_size=CONFIG['val_split'], random_state=SEED, stratify=modalities
)
print(f'Train: {len(train_records)}, Val: {len(val_records)}')

In [ ]:
# ── Dataset with synthetic symptom text pairing ──────────────────────────

IMG_MEAN = [0.485, 0.456, 0.406]
IMG_STD = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(IMG_MEAN, IMG_STD),
])

val_transform = T.Compose([
    T.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    T.ToTensor(),
    T.Normalize(IMG_MEAN, IMG_STD),
])

tokenizer = AutoTokenizer.from_pretrained(CONFIG['text_model_name'])


class MultiModalDataset(Dataset):
    def __init__(self, records, tokenizer, transform, max_length=128, seed=42):
        self.records = records
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_length = max_length
        self.rng = random.Random(seed)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        # Image
        img = Image.open(rec['image_path']).convert('RGB')
        img = self.transform(img)
        # Synthetic symptom text
        disease = self.rng.choice(rec['diseases'])
        text = generate_symptom_text(disease, self.rng)
        encoded = self.tokenizer(
            text, max_length=self.max_length, padding='max_length',
            truncation=True, return_tensors='pt',
        )
        return {
            'image': img,
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'labels': torch.tensor(rec['labels'], dtype=torch.float32),
        }


train_dataset = MultiModalDataset(train_records, tokenizer, train_transform, CONFIG['max_text_length'])
val_dataset = MultiModalDataset(val_records, tokenizer, val_transform, CONFIG['max_text_length'], seed=99)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,
                           num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
                         num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

## 5. Load Pre-trained Encoders & Build Multimodal Model

In [ ]:
# ── Build encoders ────────────────────────────────────────────────────────
image_encoder = ImageEncoder(
    backbone=CONFIG['image_backbone'],
    feature_dim=CONFIG['image_feature_dim'],
    projection_dim=CONFIG['projection_dim'],
    dropout=CONFIG['dropout'],
    pretrained=True,
)

text_encoder = TextEncoder(
    model_name=CONFIG['text_model_name'],
    hidden_dim=CONFIG['text_hidden_dim'],
    projection_dim=CONFIG['projection_dim'],
    dropout=CONFIG['dropout'],
    max_length=CONFIG['max_text_length'],
)

# ── Load pre-trained weights ──────────────────────────────────────────────
img_enc_path = os.path.join(CONFIG['image_pretrained_dir'], 'image_encoder.pth')
if os.path.exists(img_enc_path):
    state = torch.load(img_enc_path, map_location='cpu')
    image_encoder.load_state_dict(state, strict=False)
    print(f'Loaded image encoder from {img_enc_path}')
else:
    print('WARNING: No pre-trained image encoder found; using ImageNet weights.')

txt_enc_path = os.path.join(CONFIG['text_pretrained_dir'], 'text_encoder.pth')
if os.path.exists(txt_enc_path):
    state = torch.load(txt_enc_path, map_location='cpu')
    text_encoder.load_state_dict(state, strict=False)
    print(f'Loaded text encoder from {txt_enc_path}')
else:
    print('WARNING: No pre-trained text encoder found; using BioBERT defaults.')

# ── Build fusion ──────────────────────────────────────────────────────────
fusion = CrossAttentionFusion(
    dim=CONFIG['projection_dim'],
    num_heads=CONFIG['fusion_num_heads'],
    num_layers=CONFIG['fusion_num_cross_layers'],
    num_classes=CONFIG['num_classes'],
    dropout=CONFIG['dropout'],
)

model = MultiModalDetector(
    image_encoder, text_encoder, fusion,
    num_classes=CONFIG['num_classes'],
    projection_dim=CONFIG['projection_dim'],
    dropout=CONFIG['dropout'],
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal params: {total_params:,}  Trainable: {trainable_params:,}')

## 6. Training

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, device, mode='multimodal'):
    model.train()
    total_loss = 0.0
    all_preds, all_targets = [], []

    for batch in tqdm(loader, desc=f'Train({mode})', leave=False):
        images = batch['image'].to(device, non_blocking=True)
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)

        optimizer.zero_grad()
        with autocast():
            if mode == 'multimodal':
                logits = model(images, input_ids, attention_mask, mode='multimodal')
            elif mode == 'image_only':
                logits = model(image=images, mode='image_only')
            else:
                logits = model(input_ids=input_ids, attention_mask=attention_mask, mode='text_only')
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()
        all_preds.append(torch.sigmoid(logits).detach().cpu().numpy())
        all_targets.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_targets, (all_preds >= 0.5).astype(int), average='macro', zero_division=0)
    return avg_loss, f1


@torch.no_grad()
def validate(model, loader, criterion, device, mode='multimodal'):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []

    for batch in tqdm(loader, desc=f'Val({mode})', leave=False):
        images = batch['image'].to(device, non_blocking=True)
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)

        with autocast():
            if mode == 'multimodal':
                logits = model(images, input_ids, attention_mask, mode='multimodal')
            elif mode == 'image_only':
                logits = model(image=images, mode='image_only')
            else:
                logits = model(input_ids=input_ids, attention_mask=attention_mask, mode='text_only')
            loss = criterion(logits, labels)

        total_loss += loss.item()
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_targets.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_targets, (all_preds >= 0.5).astype(int), average='macro', zero_division=0)

    valid_cols = [c for c in range(all_targets.shape[1])
                  if 0 < all_targets[:, c].sum() < len(all_targets)]
    roc_auc = roc_auc_score(
        all_targets[:, valid_cols], all_preds[:, valid_cols], average='macro'
    ) if valid_cols else 0.0

    return avg_loss, f1, roc_auc, all_preds, all_targets

In [ ]:
# ── Optimizer with differential learning rates ────────────────────────────

criterion = FocalLoss(alpha=CONFIG['focal_alpha'], gamma=CONFIG['focal_gamma'])
scaler = GradScaler()

optimizer = torch.optim.AdamW([
    {'params': model.fusion.parameters(), 'lr': CONFIG['lr_fusion']},
    {'params': model.image_head.parameters(), 'lr': CONFIG['lr_fusion']},
    {'params': model.text_head.parameters(), 'lr': CONFIG['lr_fusion']},
    {'params': model.image_encoder.parameters(), 'lr': CONFIG['lr_encoder']},
    {'params': model.text_encoder.parameters(), 'lr': CONFIG['lr_encoder']},
], weight_decay=CONFIG['weight_decay'])

total_steps = len(train_loader) * CONFIG['epochs']
warmup_steps = int(total_steps * CONFIG['warmup_ratio'])
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

# ── Train multimodal fusion ───────────────────────────────────────────────
print('='*60)
print('Training Multimodal Cross-Attention Fusion')
print('='*60)

best_val_f1 = 0.0
history = []

for epoch in range(CONFIG['epochs']):
    train_loss, train_f1 = train_one_epoch(
        model, train_loader, criterion, optimizer, scheduler, scaler, DEVICE, 'multimodal'
    )
    val_loss, val_f1, val_auc, _, _ = validate(
        model, val_loader, criterion, DEVICE, 'multimodal'
    )
    history.append({
        'epoch': epoch + 1, 'train_loss': train_loss, 'train_f1': train_f1,
        'val_loss': val_loss, 'val_f1': val_f1, 'val_auc': val_auc,
    })
    print(
        f'Epoch {epoch+1}/{CONFIG["epochs"]} | '
        f'Train Loss: {train_loss:.4f} F1: {train_f1:.4f} | '
        f'Val Loss: {val_loss:.4f} F1: {val_f1:.4f} AUC: {val_auc:.4f}'
    )
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), os.path.join(CONFIG['output_dir'], 'fusion_model_best.pth'))
        print(f'  -> New best model saved (F1={val_f1:.4f})')

print(f'\nBest multimodal F1: {best_val_f1:.4f}')

## 7. ConcatFusion Baseline

In [ ]:
# ── Train a simple ConcatFusion baseline for comparison ──────────────────

concat_fusion = ConcatFusion(
    dim=CONFIG['projection_dim'],
    num_classes=CONFIG['num_classes'],
    dropout=CONFIG['dropout'],
).to(DEVICE)

# Use frozen encoders for concat baseline
concat_optimizer = torch.optim.AdamW(concat_fusion.parameters(), lr=CONFIG['lr_fusion'],
                                      weight_decay=CONFIG['weight_decay'])
concat_scheduler = get_linear_schedule_with_warmup(
    concat_optimizer, warmup_steps, len(train_loader) * 10,
)

print('='*60)
print('Training ConcatFusion Baseline (5 epochs)')
print('='*60)

concat_history = []
best_concat_f1 = 0.0

for epoch in range(5):
    # Train
    concat_fusion.train()
    model.image_encoder.eval()
    model.text_encoder.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []

    for batch in tqdm(train_loader, desc=f'Concat Ep{epoch+1}', leave=False):
        images = batch['image'].to(DEVICE, non_blocking=True)
        input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
        attention_mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
        labels = batch['labels'].to(DEVICE, non_blocking=True)

        concat_optimizer.zero_grad()
        with autocast():
            with torch.no_grad():
                img_emb = model.image_encoder(images)
                txt_emb = model.text_encoder(input_ids, attention_mask)
            logits = concat_fusion(img_emb, txt_emb)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(concat_optimizer)
        scaler.update()
        concat_scheduler.step()

        total_loss += loss.item()
        all_preds.append(torch.sigmoid(logits).detach().cpu().numpy())
        all_targets.append(labels.cpu().numpy())

    # Val
    concat_fusion.eval()
    val_preds, val_targets = [], []
    val_loss_total = 0.0
    with torch.no_grad():
        for batch in val_loader:
            images = batch['image'].to(DEVICE)
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            with autocast():
                img_emb = model.image_encoder(images)
                txt_emb = model.text_encoder(input_ids, attention_mask)
                logits = concat_fusion(img_emb, txt_emb)
                loss = criterion(logits, labels)
            val_loss_total += loss.item()
            val_preds.append(torch.sigmoid(logits).cpu().numpy())
            val_targets.append(labels.cpu().numpy())

    vp = np.concatenate(val_preds)
    vt = np.concatenate(val_targets)
    vf1 = f1_score(vt, (vp >= 0.5).astype(int), average='macro', zero_division=0)
    concat_history.append({'epoch': epoch + 1, 'val_f1': vf1})
    print(f'  Concat Epoch {epoch+1}/5 | Val F1: {vf1:.4f}')
    best_concat_f1 = max(best_concat_f1, vf1)

print(f'\nBest ConcatFusion F1: {best_concat_f1:.4f}')

## 8. 3-Way Comparison

In [ ]:
# ── Load best multimodal model ───────────────────────────────────────────
best_path = os.path.join(CONFIG['output_dir'], 'fusion_model_best.pth')
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    print('Loaded best fusion model.')

# ── Evaluate all 3 modes ──────────────────────────────────────────────────
results = {}

for mode_name in ['multimodal', 'image_only', 'text_only']:
    _, f1, auc, preds, targets = validate(model, val_loader, criterion, DEVICE, mode_name)
    results[mode_name] = {'f1': f1, 'auc': auc}
    print(f'{mode_name:15s} | F1: {f1:.4f} | AUC: {auc:.4f}')

# Add concat baseline
results['concat_fusion'] = {'f1': best_concat_f1, 'auc': 0.0}  # AUC not tracked for baseline

# ── Comparison Table ──────────────────────────────────────────────────────
print('\n' + '='*55)
print(f'{"Method":<20s} | {"Macro F1":>10s} | {"ROC-AUC":>10s}')
print('-'*55)
for method, scores in results.items():
    print(f'{method:<20s} | {scores["f1"]:>10.4f} | {scores["auc"]:>10.4f}')
print('='*55)

In [ ]:
# ── Per-modality breakdown ────────────────────────────────────────────────

_, _, _, mm_preds, mm_targets = validate(model, val_loader, criterion, DEVICE, 'multimodal')

print('\nPer-modality Multimodal Results:')
print('-' * 50)

for modality, (start, end) in CONFIG['modality_index_ranges'].items():
    mod_preds = mm_preds[:, start:end]
    mod_targets = mm_targets[:, start:end]
    active = mod_targets.sum(axis=1) > 0
    if active.sum() == 0:
        print(f'  {modality}: No samples')
        continue
    mod_f1 = f1_score(mod_targets[active], (mod_preds[active] >= 0.5).astype(int),
                      average='macro', zero_division=0)
    valid_cols = [c for c in range(mod_targets.shape[1])
                  if 0 < mod_targets[active, c].sum() < active.sum()]
    if valid_cols:
        mod_auc = roc_auc_score(mod_targets[active][:, valid_cols],
                                 mod_preds[active][:, valid_cols], average='macro')
    else:
        mod_auc = 0.0
    print(f'  {modality:10s} | F1: {mod_f1:.4f} | AUC: {mod_auc:.4f} | n={int(active.sum())}')

## 9. Save Models

In [ ]:
output_dir = CONFIG['output_dir']

# ── Save full model ───────────────────────────────────────────────────────
torch.save(model.state_dict(), os.path.join(output_dir, 'fusion_model.pth'))

# ── Save individual components for flexible loading ───────────────────────
torch.save(model.image_encoder.state_dict(), os.path.join(output_dir, 'image_encoder_fused.pth'))
torch.save(model.text_encoder.state_dict(), os.path.join(output_dir, 'text_encoder_fused.pth'))
torch.save(model.fusion.state_dict(), os.path.join(output_dir, 'cross_attention_fusion.pth'))
torch.save(model.image_head.state_dict(), os.path.join(output_dir, 'image_head_fused.pth'))
torch.save(model.text_head.state_dict(), os.path.join(output_dir, 'text_head_fused.pth'))

# ── Save tokenizer ────────────────────────────────────────────────────────
tokenizer_dir = os.path.join(output_dir, 'tokenizer')
os.makedirs(tokenizer_dir, exist_ok=True)
tokenizer.save_pretrained(tokenizer_dir)

# ── Config JSON ───────────────────────────────────────────────────────────
multimodal_config = {
    'image_backbone': CONFIG['image_backbone'],
    'image_feature_dim': CONFIG['image_feature_dim'],
    'image_size': CONFIG['image_size'],
    'text_model_name': CONFIG['text_model_name'],
    'text_hidden_dim': CONFIG['text_hidden_dim'],
    'projection_dim': CONFIG['projection_dim'],
    'dropout': CONFIG['dropout'],
    'num_classes': CONFIG['num_classes'],
    'fusion_type': CONFIG['fusion_type'],
    'fusion_num_heads': CONFIG['fusion_num_heads'],
    'fusion_num_cross_layers': CONFIG['fusion_num_cross_layers'],
    'all_labels': CONFIG['all_labels'],
    'modality_index_ranges': CONFIG['modality_index_ranges'],
    'training_results': results,
    'best_multimodal_f1': best_val_f1,
    'training_history': history,
}

with open(os.path.join(output_dir, 'multimodal_config.json'), 'w') as f:
    json.dump(multimodal_config, f, indent=2)

# ── Training history plot ─────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_x = [h['epoch'] for h in history]

axes[0].plot(epochs_x, [h['train_loss'] for h in history], label='Train')
axes[0].plot(epochs_x, [h['val_loss'] for h in history], label='Val')
axes[0].legend(); axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')

axes[1].plot(epochs_x, [h['val_f1'] for h in history], 'g-')
axes[1].set_title('Val Macro F1'); axes[1].set_xlabel('Epoch')

axes[2].plot(epochs_x, [h['val_auc'] for h in history], 'm-')
axes[2].set_title('Val ROC-AUC'); axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'multimodal_training_history.png'), dpi=150)
plt.show()

# List outputs
print('\nSaved files:')
for fname in sorted(os.listdir(output_dir)):
    fpath = os.path.join(output_dir, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f'  {fname:50s} {size_mb:.1f} MB')